In [1]:
import geopandas as gpd
import os 
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import concurrent.futures
import xarray as xr

In [ ]:
def read_geoparquet(file):
    try:
        return gpd.read_parquet(file)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        return None

def combine_geoparquets(input_folder, output_file):
    # Ensure input folder exists
    input_folder = Path(input_folder)
    if not input_folder.is_dir():
        raise ValueError(f"Input folder {input_folder} does not exist")

    # Get list of all .parquet files in the folder
    parquet_files = list(input_folder.glob("*.parquet"))
    if not parquet_files:
        raise ValueError(f"No GeoParquet files found in {input_folder}")

    # Read all GeoParquet files in parallel with progress bar
    gdfs = []
    with concurrent.futures.ProcessPoolExecutor() as executor:
        for gdf in tqdm(executor.map(read_geoparquet, parquet_files), total=len(parquet_files), desc="Processing GeoParquet files"):
            if gdf is not None:
                gdfs.append(gdf)

    if not gdfs:
        raise ValueError("No valid GeoParquet files could be read")

    # Concatenate all GeoDataFrames
    full_data = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

    # Write to output GeoParquet file
    output_file = Path(output_file)
    full_data.to_parquet(output_file, index=False)
    print(f"Combined GeoParquet saved to {output_file}")

if __name__ == "__main__":
    # Example usage
    input_folder = "./ISA_2024_Raw_Yields_chunks"
    output_file = f"./Yield_{input_folder.split('_')[1]}.parquet"
    combine_geoparquets(input_folder, output_file)

Processing GeoParquet files: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3203/3203 [00:09<00:00, 340.84it/s]


Combined GeoParquet saved to Yield_2024.parquet


In [3]:
full_data = gpd.read_parquet('Yield_2024.parquet')
full_data = full_data.iloc[:,1:]
full_data = full_data.drop_duplicates(subset='geometry', keep='first')

In [4]:
full_data

,Layer_ID,Yld_Vol_Dr,Moisture,Speed_mph,x,y,Time,geometry
0,ST2024IA0144,296.149,13.11,1.42,-92.037987,41.366707,2024-10-11,POINT (-92.03799 41.36671)
1,ST2024IA0144,307.531,13.19,2.04,-92.037987,41.366722,2024-10-11,POINT (-92.03799 41.36672)
2,ST2024IA0144,213.966,13.13,2.37,-92.037987,41.366737,2024-10-11,POINT (-92.03799 41.36674)
3,ST2024IA0144,266.620,13.17,2.20,-92.037987,41.366756,2024-10-11,POINT (-92.03799 41.36676)
4,ST2024IA0144,319.009,13.11,1.96,-92.037987,41.366772,2024-10-11,POINT (-92.03799 41.36677)
...,...,...,...,...,...,...,...,...
3202694,ST2024IA0080,289.978,21.26,2.93,-93.370644,42.831043,2024-09-16,POINT (-93.37064 42.83104)
3202695,ST2024IA0080,264.744,21.01,2.93,-93.370628,42.831039,2024-09-16,POINT (-93.37063 42.83104)
3202696,ST2024IA0080,278.867,20.75,2.78,-93.370613,42.831039,2024-09-16,POINT (-93.37061 42.83104)
3202697,ST2024IA0080,246.946,20.88,2.76,-93.370598,42.831039,2024-09-16,POINT (-93.3706 42.83104)


In [5]:
crop_class = pd.read_csv('Crop_Classification_2024.csv')
# Renaming specific columns
crop_class = crop_class.rename(columns={'X_cent': 'x', 'Y_cent': 'y'})
crop_class

,Unnamed: 0,Layer_ID,Yield,x,y,Crop
0,1,ST2024IA0008,33.320348,-92.444099,43.268354,Soybean
1,2,ST2024IA0188,42.013234,-92.242478,42.611682,Soybean
2,3,ST2024IA0185,43.639158,-92.131834,42.582971,Soybean
3,4,ST2024IA0156,44.538230,-93.193891,43.405919,Soybean
4,5,ST2024IA0069,44.976784,-94.430110,41.655763,Soybean
...,...,...,...,...,...,...
109,110,ST2024IA0136,273.310623,-91.193444,40.946191,Corn
110,111,ST2024IA0001,275.407958,-94.999541,41.003108,Corn
111,112,ST2024IA0143,276.435084,-92.040706,41.370142,Corn
112,113,ST2024IA0144,276.435084,-92.040706,41.370142,Corn


In [6]:
# Using map (efficient, handles duplicates in df1)
crop_mapping = dict(zip(crop_class['Layer_ID'], crop_class['Crop']))
full_data['Crop'] = full_data['Layer_ID'].map(crop_mapping)
full_data.dropna(subset = ['Crop'],inplace=True)
full_data

,Layer_ID,Yld_Vol_Dr,Moisture,Speed_mph,x,y,Time,geometry,Crop
0,ST2024IA0144,296.149,13.11,1.42,-92.037987,41.366707,2024-10-11,POINT (-92.03799 41.36671),Corn
1,ST2024IA0144,307.531,13.19,2.04,-92.037987,41.366722,2024-10-11,POINT (-92.03799 41.36672),Corn
2,ST2024IA0144,213.966,13.13,2.37,-92.037987,41.366737,2024-10-11,POINT (-92.03799 41.36674),Corn
3,ST2024IA0144,266.620,13.17,2.20,-92.037987,41.366756,2024-10-11,POINT (-92.03799 41.36676),Corn
4,ST2024IA0144,319.009,13.11,1.96,-92.037987,41.366772,2024-10-11,POINT (-92.03799 41.36677),Corn
...,...,...,...,...,...,...,...,...,...
3202694,ST2024IA0080,289.978,21.26,2.93,-93.370644,42.831043,2024-09-16,POINT (-93.37064 42.83104),Corn
3202695,ST2024IA0080,264.744,21.01,2.93,-93.370628,42.831039,2024-09-16,POINT (-93.37063 42.83104),Corn
3202696,ST2024IA0080,278.867,20.75,2.78,-93.370613,42.831039,2024-09-16,POINT (-93.37061 42.83104),Corn
3202697,ST2024IA0080,246.946,20.88,2.76,-93.370598,42.831039,2024-09-16,POINT (-93.3706 42.83104),Corn


In [7]:
# Filter df1 based on the conditions
full_data_filtered = full_data[
    # Keep rows where Crop is not Soybean or Yld_Vol_Dr is in range 0-150 for Soybean
    ((full_data['Crop'] != 'Soybean') | ((full_data['Yld_Vol_Dr'] >= 0) & (full_data['Yld_Vol_Dr'] <= 150))) &
    # Keep rows where Crop is not Corn or Yld_Vol_Dr is in range 0-550 for Corn
    ((full_data['Crop'] != 'Corn') | ((full_data['Yld_Vol_Dr'] >= 0) & (full_data['Yld_Vol_Dr'] <= 550)))
]
# full_data_filtered = full_data_filtered.set_index(['x','y'])
full_data_filtered.to_parquet('Yield_2024_filtered.parquet')

In [ ]:
x_data = full_data_filtered.to_xarray()
x_data